#Ingesta de carpeta con archivos JSON

In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "production_country"
v_esquema = "movie_silver"
v_tabla = "productions_countries"
v_partition = "file_date"
dbutils.widgets.text("p_esquema", v_esquema)
dbutils.widgets.text("p_tabla", v_tabla)

In [0]:
#1. Leer archivos JSON usando DataFrameReader de Spark
# Define el la estructura personName
production_country_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("countryId", IntegerType(), True)
])

# Cargamos el archivo utilizando la estructura definida
production_country_df = spark.read\
    .schema(production_country_schema)\
    .option("multiLine", "true")\
    .json(f"{bronze_folder_path}/{v_file_date}/{v_archivo}")

# Mostramos el resultado
display(production_country_df.filter(col("movieId") == 5))


In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas

production_country_renamed_df = production_country_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("countryId", "country_id")

production_country_renamed_df = add_ingestion_date(production_country_renamed_df)
production_country_renamed_df = add_env(production_country_renamed_df)
production_country_renamed_df = add_file_date (production_country_renamed_df)


display(production_country_renamed_df.limit(10))


In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
overwrite_partition (v_esquema, v_tabla, v_partition, v_file_date)

In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 
#production_country_renamed_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.productions_countries")

production_country_renamed_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
print(f"Se insertaron {production_country_renamed_df.count()} registros en la tabla {v_esquema}.{v_tabla}")


In [0]:
dbutils.notebook.exit("El notebook 12. Ingestion folder_production_country, termino correctamente")